# 02 — Silver: Data Quality

**Tickets:** I-06, I-07  
**Purpose:** Run data quality assertions on the Silver table; document assumptions and findings.

---

## Setup

In [ ]:
from src.constants import (
    MAX_FARE_AMOUNT,
    MAX_TIP_AMOUNT,
    MAX_TOTAL_AMOUNT,
    MAX_TRIP_DISTANCE,
    SILVER_TABLE,
    VALID_EXTRA_VALUES,
    VALID_RATE_CODES,
)
from src.validators import (
    check_accepted_values,
    check_column_exists,
    check_max,
    check_no_nulls,
    check_non_negative,
    check_not_empty,
    check_positive,
)

print("Setup complete")

## Read Silver table

In [ ]:
silver_df = spark.read.table(SILVER_TABLE)
print(f"Silver table: {SILVER_TABLE}")
print(f"Row count: {silver_df.count():,}")
silver_df.printSchema()

## I-06 — Data quality checks

These assertions validate that I-03 (structural cleaning) and I-04 (outlier removal) have been applied correctly. Every check must pass before Gold/ML can consume the Silver table.

**Check categories:**
1. **Structural** — table not empty, required columns exist, no NULLs in key fields
2. **Business rules** — values within expected ranges after I-04 cleaning
3. **Referential integrity** — categorical codes within valid sets

In [ ]:
print("=" * 60)
print("1. STRUCTURAL CHECKS")
print("=" * 60)

# Table is not empty
check_not_empty(silver_df, "Silver")

# Required columns exist
for col in [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type",
    "passenger_count",
    "pickup_zone",
    "dropoff_zone",
]:
    check_column_exists(silver_df, col)

# No NULLs in key fields (guaranteed by I-03 drop_corrupt_rows)
for col in [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type",
]:
    check_no_nulls(silver_df, col)

In [ ]:
print("=" * 60)
print("2. BUSINESS RULE CHECKS (I-04 guarantees)")
print("=" * 60)

# trip_distance > 0 and <= MAX (I-04a, I-04d)
check_positive(silver_df, "trip_distance")
check_max(silver_df, "trip_distance", MAX_TRIP_DISTANCE)

# fare_amount > 0 and <= MAX (I-04b, I-04d)
check_positive(silver_df, "fare_amount")
check_max(silver_df, "fare_amount", MAX_FARE_AMOUNT)

# total_amount >= 0 and <= MAX (I-04b, I-04d)
check_non_negative(silver_df, "total_amount")
check_max(silver_df, "total_amount", MAX_TOTAL_AMOUNT)

# passenger_count > 0 (I-04c)
check_positive(silver_df, "passenger_count")

# tip_amount >= 0 and <= MAX (I-04f)
check_non_negative(silver_df, "tip_amount")
check_max(silver_df, "tip_amount", MAX_TIP_AMOUNT)

# extra surcharge: only valid values or NULL (I-04f)
check_accepted_values(silver_df, "extra", list(VALID_EXTRA_VALUES))

In [ ]:
print("=" * 60)
print("3. REFERENTIAL INTEGRITY CHECKS")
print("=" * 60)

# payment_type in {1, 2, 3, 4, 5, 6}
check_accepted_values(silver_df, "payment_type", [1, 2, 3, 4, 5, 6])

# rate_code_id in valid set or NULL (code 99 mapped to NULL in I-03)
check_accepted_values(silver_df, "rate_code_id", list(VALID_RATE_CODES))

# vendor_id in {1, 2}
check_accepted_values(silver_df, "vendor_id", [1, 2])

print()
print("=" * 60)
print("ALL DATA QUALITY CHECKS PASSED")
print("=" * 60)

## I-07 — Assumptions & Data Quality Log

<!-- Document assumptions, known issues and mitigations here -->

| Column | Issue | Mitigation |
|--------|-------|------------|
| `fare_amount` | Rows with $0 or negative fare | Dropped in cleaning |
| `trip_distance` | Rows with 0 distance | Dropped in cleaning |
| | | |